In [ ]:
#import artists
duck_con.execute(f"""
COPY (
    WITH sample_artists AS (
        SELECT
            id,
            gid,
            name,
            begin_date_year,
            type,
            area,
            gender
        FROM mb_pg.musicbrainz.artist
        ORDER BY id
    )
    SELECT
        id,
        name,
        begin_date_year,
        type,
        area,
        gender
    FROM sample_artists
)
TO './data/mb_artist.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import artist tags
duck_con.execute(f"""
COPY (
    WITH artist_tag_sample AS (
        SELECT
            artist AS artist_id,
            tag AS tag_id,
            count AS tag_count
        FROM mb_pg.musicbrainz.artist_tag
        ORDER BY artist
    )
    SELECT
        artist_id,
        tag_id,
        tag_count
    FROM artist_tag_sample
)
TO './data/mb_artist_tag.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import artist ratings
duck_con.execute(f"""
COPY (
    SELECT
        id AS artist_id,
        rating,
        rating_count
    FROM mb_pg.musicbrainz.artist_meta
    WHERE rating IS NOT NULL
    ORDER BY rating_count DESC
)
TO './data/mb_artist_ratings.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import albums
duck_con.execute(f"""
COPY (
    SELECT
        id,
        gid,
        name,
        artist_credit,
        type
    FROM mb_pg.musicbrainz.release_group
    WHERE type = 1
    ORDER BY id
)
TO './data/mb_album.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album tags
duck_con.execute(f"""
COPY (
    WITH album_ids AS (
        SELECT id 
        FROM mb_pg.musicbrainz.release_group 
        WHERE type = 1
    )
    SELECT
        t.release_group AS album_id,
        t.tag AS tag_id,
        t.count AS tag_count
    FROM mb_pg.musicbrainz.release_group_tag t
    JOIN album_ids a ON t.release_group = a.id
    WHERE t.count > 0
    ORDER BY t.release_group
)
TO './data/mb_album_tag_map.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album ratings
duck_con.execute(f"""
COPY (
    SELECT
        m.id AS album_id,
        m.rating,
        m.rating_count
    FROM mb_pg.musicbrainz.release_group_meta m
    JOIN mb_pg.musicbrainz.release_group rg ON m.id = rg.id
    WHERE rg.type = 1 
      AND m.rating IS NOT NULL
    ORDER BY m.rating_count DESC
)
TO './data/mb_album_ratings.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")